# Практическая работа №2: Обработка выборочных данных. Нахождение точечных оценок параметров распределения

Выполнили студенты гр. 2384 Поглазов Никита Васильевич и Вовченко София Евгеньевна. Вариант 14.

## Цель работы

Получение практических навыков нахождения точечных статистических оценок параметров распределения.


## Основные теоретические положения

Выборочное среднее — среднее арифметическое значений выборки:
$$ \bar{x}_{\textit{в}} = \frac{1}{n} \sum_{i=1}^k m_i x_i $$

Выборочная дисперсия — мера разброса значений вокруг среднего:
$$ D_{\textit{в}} = \frac{1}{n} \sum_{i=1}^k m_i (x_i - \bar{x}_{\textit{в}})^2 = \overline{x^2} - (\bar{x}_{\textit{в}})^2 $$

Выборочное среднее квадратическое отклонение (СКО):
$$ \sigma_{\textit{в}} = \sqrt{D_{\textit{в}}} $$

**Метод условных вариант (метод произведений)**

Позволяет упростить вычисления числовых характеристик. Вводятся условные варианты:
$$ u_i = \frac{x_i - C}{h} $$
где $ C $ — ложный нуль (обычно значение середины интервала с наибольшей частотой), $ h $ — длина интервала.

Условные эмпирические моменты $ k $-го порядка:
$$ \bar{M}^*_k = \frac{1}{n} \sum_{i=1}^k m_i u_i^k $$

Связь между характеристиками исходной выборки и условными моментами:
$$ \bar{x}_{\textit{в}} = \bar{M}^*_1 h + C $$
$$ D_{\textit{в}} = (\bar{M}^*_2 - (\bar{M}^*_1)^2) h^2 $$

Выборочная дисперсия является смещенной оценкой генеральной дисперсии. Исправленная (несмещенная) дисперсия:
$$ s^2 = \frac{n}{n-1} D_{\textit{в}} $$
Исправленное СКО:
$$ s = \sqrt{s^2} $$

Центральные эмпирические моменты $ \bar{m}_k $ через условные моменты:
$$ \bar{m}_2 = (\bar{M}^*_2 - (\bar{M}^*_1)^2) h^2 $$
$$ \bar{m}_3 = (\bar{M}^*_3 - 3 \bar{M}^*_2 \bar{M}^*_1 + 2 (\bar{M}^*_1)^3) h^3 $$
$$ \bar{m}_4 = (\bar{M}^*_4 - 4 \bar{M}^*_3 \bar{M}^*_1 + 6 \bar{M}^*_2 (\bar{M}^*_1)^2 - 3 (\bar{M}^*_1)^4) h^4 $$

Коэффициент асимметрии $ \bar{A}_s $ (характеризует скошенность распределения):
$$ \bar{A}_s = \frac{\bar{m}_3}{\sigma_{\textit{в}}^3} $$
*   $ \bar{A}_s > 0 $ — правосторонняя асимметрия.
*   $ \bar{A}_s < 0 $ — левосторонняя асимметрия.

Эксцесс $ \bar{E} $ (характеризует островершинность распределения):
$$ \bar{E} = \frac{\bar{m}_4}{\sigma_{\textit{в}}^4} - 3 $$
*   $ \bar{E} > 0 $ — островершинное распределение.
*   $ \bar{E} < 0 $ — плосковершинное распределение.

Мода $ M^*_o $ (значение, встречающееся наиболее часто):
$$ M^*_o = x_{Mo}^{(0)} + h \frac{m_{Mo} - m_{Mo-1}}{(m_{Mo} - m_{Mo-1}) + (m_{Mo} - m_{Mo+1})} $$
где $ x_{Mo}^{(0)} $ — нижняя граница модального интервала.

Медиана $ M^*_e $ (значение, делящее вариационный ряд пополам):
$$ M^*_e = x_{Me}^{(0)} + h \frac{0.5n - S_{Me-1}}{m_{Me}} $$
где $ x_{Me}^{(0)} $ — нижняя граница медианного интервала, $ S_{Me-1} $ — накопленная частота интервала, предшествующего медианному.

Коэффициент вариации (мера относительного разброса):
$$ V^* = \frac{\sigma_{\textit{в}}}{\bar{x}_{\textit{в}}} \cdot 100\% $$
*   Если $ V^* \le 33\% $, совокупность считается однородной.

## Постановка задачи

Для заданных выборочных данных вычислить с использованием метода моментов и условных вариант точечные статистические оценки математического ожидания, дисперсии, среднеквадратичного отклонения, асимметрии, эксцесса, моды, медианы и коэффициента вариации исследуемой случайной величины. Полученные результаты содержательно проинтерпретировать.


## Выполнение работы


In [52]:
import numpy as np
import pandas as pd
from IPython.display import display_markdown

from math import ceil, log10

In [53]:
df = pd.read_csv('../samples.csv')

In [54]:
length_ranged = df.SepalLengthCm.sort_values().to_numpy()
width_ranged = df.SepalWidthCm.sort_values().to_numpy()

In [55]:
def create_interval_series(data: np.ndarray) -> pd.DataFrame:
    """
    Creates an interval series distribution from the given data.
    """
    n = len(data)
    intervals_number = ceil(1 + 3.322 * log10(n))
    data_min, data_max = data[0], data[-1]
    
    interval_size = (data_max - data_min) / intervals_number

    bins = data_min + np.array([i * interval_size for i in range(intervals_number + 1)])
    bins[-1] += 1e-10 # Ensure the maximum value is included

    cuts = pd.cut(data, bins=bins, right=False)
    interval_counts = cuts.value_counts().sort_index().to_numpy()

    interval_centers = (bins[:-1] + bins[1:]) / 2

    df_intervals = pd.DataFrame(
        {
            "i": np.arange(1, intervals_number + 1),
            "interval": [
                f"[{bins[i]:.2f}; {bins[i + 1]:.2f})" for i in range(intervals_number)
            ],
            "x_i": interval_centers,
            "m_i": interval_counts,
        }
    )
    
    df_intervals["w_i"] = interval_counts / n
    df_intervals["m_i_cum"] = interval_counts.cumsum()
    df_intervals["w_i_cum"] = df_intervals["w_i"].cumsum()
    
    return df_intervals

def display_interval_table(df_intervals: pd.DataFrame) -> None:
    """
    Displays the interval series DataFrame with LaTeX headers.
    """
    display_df = df_intervals.copy()
    
    display_df["x_i"] = display_df["x_i"].round(2)

    sum_row = pd.Series(
        {
            "i": "Σ",
            "interval": "",
            "x_i": "",
            "m_i": display_df["m_i"].sum(),
            "w_i": display_df["w_i"].sum(),
            "m_i_cum": "-",
            "w_i_cum": "-",
        }
    )
    
    display_df = pd.concat([display_df, sum_row.to_frame().T], ignore_index=True)

    display_df.columns = [
        "$i$",
        "$[x_i; x_{i+1})$",
        r"$\tilde{x}_i$",
        "$m_i$",
        r"$\tilde{w}_i$",
        r"$m_i^{нак}$",
        r"$\tilde{w}_i^{нак}$",
    ]
    
    display_markdown(display_df.to_markdown(), raw=True)

In [56]:
length_intervals = create_interval_series(length_ranged)
display_interval_table(length_intervals)

|    | $i$   | $[x_i; x_{i+1})$   | $\tilde{x}_i$   |   $m_i$ |   $\tilde{w}_i$ | $m_i^{нак}$   | $\tilde{w}_i^{нак}$   |
|---:|:------|:-------------------|:----------------|--------:|----------------:|:--------------|:----------------------|
|  0 | 1     | [4.30; 4.75)       | 4.53            |       7 |       0.0654206 | 7             | 0.06542056074766354   |
|  1 | 2     | [4.75; 5.20)       | 4.97            |      19 |       0.17757   | 26            | 0.24299065420560745   |
|  2 | 3     | [5.20; 5.65)       | 5.43            |      18 |       0.168224  | 44            | 0.41121495327102797   |
|  3 | 4     | [5.65; 6.10)       | 5.88            |      17 |       0.158879  | 61            | 0.5700934579439252    |
|  4 | 5     | [6.10; 6.55)       | 6.32            |      23 |       0.214953  | 84            | 0.7850467289719626    |
|  5 | 6     | [6.55; 7.00)       | 6.78            |      10 |       0.0934579 | 94            | 0.8785046728971962    |
|  6 | 7     | [7.00; 7.45)       | 7.22            |       7 |       0.0654206 | 101           | 0.9439252336448598    |
|  7 | 8     | [7.45; 7.90)       | 7.68            |       6 |       0.0560748 | 107           | 1.0                   |
|  8 | Σ     |                    |                 |     107 |       1         | -             | -                     |

In [57]:
width_intervals = create_interval_series(width_ranged)
display_interval_table(width_intervals)

|    | $i$   | $[x_i; x_{i+1})$   | $\tilde{x}_i$   |   $m_i$ |   $\tilde{w}_i$ | $m_i^{нак}$   | $\tilde{w}_i^{нак}$   |
|---:|:------|:-------------------|:----------------|--------:|----------------:|:--------------|:----------------------|
|  0 | 1     | [2.00; 2.30)       | 2.15            |       4 |       0.0373832 | 4             | 0.037383177570093455  |
|  1 | 2     | [2.30; 2.60)       | 2.45            |       9 |       0.0841121 | 13            | 0.12149532710280372   |
|  2 | 3     | [2.60; 2.90)       | 2.75            |      31 |       0.28972   | 44            | 0.41121495327102797   |
|  3 | 4     | [2.90; 3.20)       | 3.05            |      23 |       0.214953  | 67            | 0.6261682242990654    |
|  4 | 5     | [3.20; 3.50)       | 3.35            |      23 |       0.214953  | 90            | 0.8411214953271028    |
|  5 | 6     | [3.50; 3.80)       | 3.65            |      11 |       0.102804  | 101           | 0.9439252336448598    |
|  6 | 7     | [3.80; 4.10)       | 3.95            |       4 |       0.0373832 | 105           | 0.9813084112149533    |
|  7 | 8     | [4.10; 4.40)       | 4.25            |       2 |       0.0186916 | 107           | 1.0                   |
|  8 | Σ     |                    |                 |     107 |       1         | -             | -                     |

  - Для середин интервального ряда, полученного в практической работе №1, вычислить условные варианты. Заполнить табл. 1 (*в последней строке $ \Sigma $ необходимо заполнить суммы столбцов; ячейки отмеченные прочерком заполнять не надо*). Провести контроль вычислений.

In [58]:
def calculate_conditional_moments(df_intervals: pd.DataFrame) -> tuple[pd.DataFrame, float, float]:
    """
    Calculates conditional variants and moments for the given interval series.
    """
    df = df_intervals.copy()
    
    max_freq_idx = df['m_i'].idxmax()
    C = pd.to_numeric(df.loc[max_freq_idx, 'x_i'])
    h = df['x_i'][1] - df['x_i'][0]
    
    df['u_i'] = ((df['x_i'] - C) / h).round().astype(int)
    
    df['m_i_u_i'] = df['m_i'] * df['u_i']
    df['m_i_u_i_2'] = df['m_i'] * (df['u_i'] ** 2)
    df['m_i_u_i_3'] = df['m_i'] * (df['u_i'] ** 3)
    df['m_i_u_i_4'] = df['m_i'] * (df['u_i'] ** 4)
    df['m_i_u_i_plus_1_4'] = df['m_i'] * ((df['u_i'] + 1) ** 4)
    
    return df, C, h

def display_conditional_moments_table(df_moments: pd.DataFrame) -> None:
    """
    Displays the conditional moments table with sums and checks.
    """
    display_df = df_moments[["x_i", "m_i", "u_i", "m_i_u_i", "m_i_u_i_2", "m_i_u_i_3", "m_i_u_i_4", "m_i_u_i_plus_1_4"]].copy()

    display_df["x_i"] = display_df["x_i"].round(3)
    
    sums = display_df.sum()
    sum_row = pd.DataFrame([sums], columns=display_df.columns)
    sum_row.index = ["Σ"]
    
    sum_row["x_i"] = "-"
    sum_row["u_i"] = "-"
    
    display_df = pd.concat([display_df, sum_row])
    
    display_df.columns = [
        r"$\tilde{x}_i$",
        "$m_i$",
        "$u_i$",
        "$m_i u_i$",
        "$m_i u_i^2$",
        "$m_i u_i^3$",
        "$m_i u_i^4$",
        "$m_i (u_i + 1)^4$"
    ]
    
    display_markdown(display_df.to_markdown(), raw=True)
    
    sum_m_u_4 = sums["m_i_u_i_4"]
    sum_m_u_3 = sums["m_i_u_i_3"]
    sum_m_u_2 = sums["m_i_u_i_2"]
    sum_m_u = sums["m_i_u_i"]
    sum_m = sums["m_i"]
    sum_m_u_plus_1_4 = sums["m_i_u_i_plus_1_4"]
    
    right_side = sum_m_u_4 + 4 * sum_m_u_3 + 6 * sum_m_u_2 + 4 * sum_m_u + sum_m
    
    print(f"Check sum: {sum_m_u_plus_1_4} == {right_side}")
    if np.isclose(sum_m_u_plus_1_4, right_side):
        print("Calculation control passed!")
    else:
        print("Calculation control FAILED!")

In [59]:
length_moments, length_C, length_h = calculate_conditional_moments(length_intervals)
print(f"Selected C = {length_C}, h = {length_h:.4f}")
display_conditional_moments_table(length_moments)


Selected C = 6.325, h = 0.4500


|    | $\tilde{x}_i$   |   $m_i$ | $u_i$   |   $m_i u_i$ |   $m_i u_i^2$ |   $m_i u_i^3$ |   $m_i u_i^4$ |   $m_i (u_i + 1)^4$ |
|:---|:----------------|--------:|:--------|------------:|--------------:|--------------:|--------------:|--------------------:|
| 0  | 4.525           |       7 | -4      |         -28 |           112 |          -448 |          1792 |                 567 |
| 1  | 4.975           |      19 | -3      |         -57 |           171 |          -513 |          1539 |                 304 |
| 2  | 5.425           |      18 | -2      |         -36 |            72 |          -144 |           288 |                  18 |
| 3  | 5.875           |      17 | -1      |         -17 |            17 |           -17 |            17 |                   0 |
| 4  | 6.325           |      23 | 0       |           0 |             0 |             0 |             0 |                  23 |
| 5  | 6.775           |      10 | 1       |          10 |            10 |            10 |            10 |                 160 |
| 6  | 7.225           |       7 | 2       |          14 |            28 |            56 |           112 |                 567 |
| 7  | 7.675           |       6 | 3       |          18 |            54 |           162 |           486 |                1536 |
| Σ  | -               |     107 | -       |         -96 |           464 |          -894 |          4244 |                3175 |

Check sum: 3175.0 == 3175.0
Calculation control passed!


In [60]:
width_moments, width_C, width_h = calculate_conditional_moments(width_intervals)
print(f"Selected C = {width_C}, h = {width_h:.4f}")
display_conditional_moments_table(width_moments)

Selected C = 2.75, h = 0.3000


|    | $\tilde{x}_i$   |   $m_i$ | $u_i$   |   $m_i u_i$ |   $m_i u_i^2$ |   $m_i u_i^3$ |   $m_i u_i^4$ |   $m_i (u_i + 1)^4$ |
|:---|:----------------|--------:|:--------|------------:|--------------:|--------------:|--------------:|--------------------:|
| 0  | 2.15            |       4 | -2      |          -8 |            16 |           -32 |            64 |                   4 |
| 1  | 2.45            |       9 | -1      |          -9 |             9 |            -9 |             9 |                   0 |
| 2  | 2.75            |      31 | 0       |           0 |             0 |             0 |             0 |                  31 |
| 3  | 3.05            |      23 | 1       |          23 |            23 |            23 |            23 |                 368 |
| 4  | 3.35            |      23 | 2       |          46 |            92 |           184 |           368 |                1863 |
| 5  | 3.65            |      11 | 3       |          33 |            99 |           297 |           891 |                2816 |
| 6  | 3.95            |       4 | 4       |          16 |            64 |           256 |          1024 |                2500 |
| 7  | 4.25            |       2 | 5       |          10 |            50 |           250 |          1250 |                2592 |
| Σ  | -               |     107 | -       |         111 |           353 |           969 |          3629 |               10174 |

Check sum: 10174.0 == 10174.0
Calculation control passed!


  - Вычислить условные эмпирические моменты $ \bar{M}^*_k $ через условные варианты. С помощью условных эмпирических моментов вычислить центральные эмпирические моменты $ \bar{m}_k $. Полученные результаты занести в табл. 2.


In [61]:
def calculate_moments_metrics(df_moments: pd.DataFrame, h: float) -> pd.DataFrame:
    """
    Calculates conditional empirical moments and central empirical moments from the moments DataFrame.
    """
    n = df_moments["m_i"].sum()
    
    M_star = {}
    M_star[1] = df_moments["m_i_u_i"].sum() / n
    M_star[2] = df_moments["m_i_u_i_2"].sum() / n
    M_star[3] = df_moments["m_i_u_i_3"].sum() / n
    M_star[4] = df_moments["m_i_u_i_4"].sum() / n
    
    mu = {}
    mu[1] = 0.0
    mu[2] = M_star[2] - M_star[1]**2
    mu[3] = M_star[3] - 3 * M_star[2] * M_star[1] + 2 * M_star[1]**3
    mu[4] = M_star[4] - 4 * M_star[3] * M_star[1] + 6 * M_star[2] * M_star[1]**2 - 3 * M_star[1]**4
    
    m_star = {}
    m_star[1] = mu[1] * h
    m_star[2] = mu[2] * h**2
    m_star[3] = mu[3] * h**3
    m_star[4] = mu[4] * h**4
    
    results: dict[str, list[float | int]] = {
        "k": [1, 2, 3, 4],
        "M*": [M_star[1], M_star[2], M_star[3], M_star[4]],
        "m*": [m_star[1], m_star[2], m_star[3], m_star[4]]
    }
    
    return pd.DataFrame(results)

def display_moments_metrics(df_metrics: pd.DataFrame) -> None:
    display_df = df_metrics.copy()
    display_df.columns = [
        "$k$",
        r"$\bar{M}^*_k$",
        r"$\bar{m}_k$"
    ]
    display_markdown(display_df.to_markdown(index=False), raw=True)

In [62]:
moments_metrics_length = calculate_moments_metrics(length_moments, length_h)
display_moments_metrics(moments_metrics_length)

|   $k$ |   $\bar{M}^*_k$ |   $\bar{m}_k$ |
|------:|----------------:|--------------:|
|     1 |       -0.897196 |      0        |
|     2 |        4.33645  |      0.715126 |
|     3 |       -8.35514  |      0.170621 |
|     4 |       39.6636   |      1.17601  |

In [63]:
moments_metrics_width = calculate_moments_metrics(width_moments, width_h)
display_moments_metrics(moments_metrics_width)

|   $k$ |   $\bar{M}^*_k$ |   $\bar{m}_k$ |
|------:|----------------:|--------------:|
|     1 |         1.03738 |     0         |
|     2 |         3.29907 |     0.200061  |
|     3 |         9.05607 |     0.0275853 |
|     4 |        33.9159  |     0.114737  |

  - Вычислить выборочные среднее $ \bar x_{\textit{в}} $ и дисперсию $ D_{\textit{в}} $ с помощью стандартной формулы и с помощью условных вариант. Убедиться, что результаты совпадают. Вычислить выборочное СКО $ \sigma_{\textit{в}} $.


In [64]:
def calculate_sample_stats(df_moments: pd.DataFrame, h: float, C: float) -> dict[str, float]:
    """
    Calculates sample mean, variance, and standard deviation using standard formulas and conditional variants.
    """
    n = df_moments["m_i"].sum()
    
    sum_m_x = (df_moments["m_i"] * df_moments["x_i"]).sum()
    x_bar_std = sum_m_x / n
    
    sum_m_x_diff_sq = (df_moments["m_i"] * (df_moments["x_i"] - x_bar_std)**2).sum()
    D_v_std = sum_m_x_diff_sq / n
    
    sigma_v_std = np.sqrt(D_v_std)
    
    M_star_1 = df_moments["m_i_u_i"].sum() / n
    M_star_2 = df_moments["m_i_u_i_2"].sum() / n
    
    x_bar_cond = M_star_1 * h + C
    
    D_v_cond = (M_star_2 - M_star_1**2) * (h**2)
    
    sigma_v_cond = np.sqrt(D_v_cond)
    
    return {
        "x_bar_std": x_bar_std,
        "D_v_std": D_v_std,
        "sigma_v_std": sigma_v_std,
        "x_bar_cond": x_bar_cond,
        "D_v_cond": D_v_cond,
        "sigma_v_cond": sigma_v_cond
    }

def display_sample_stats(stats: dict[str, float]) -> None:
    data = [
        [r"Выборочное среднее $\bar{x}_{\textit{в}}$", stats['x_bar_std'], stats['x_bar_cond']],
        [r"Выборочная дисперсия $D_{\textit{в}}$", stats['D_v_std'], stats['D_v_cond']],
        [r"Выборочное СКО $\sigma_{\textit{в}}$", stats['sigma_v_std'], stats['sigma_v_cond']],
    ]
    
    df_display = pd.DataFrame(data, columns=["Характеристика", "Стандартный метод", "Метод условных вариант"])
    
    df_display["Совпадение"] = np.isclose(df_display["Стандартный метод"], df_display["Метод условных вариант"])
    
    display_markdown(df_display.to_markdown(index=False), raw=True)

In [ ]:
stats = calculate_sample_stats(length_moments, length_h, length_C)
display_sample_stats(stats)

| Характеристика                            |   Стандартный метод |   Метод условных вариант | Совпадение   |
|:------------------------------------------|--------------------:|-------------------------:|:-------------|
| Выборочное среднее $\bar{x}_{\textit{в}}$ |            5.92126  |                 5.92126  | True         |
| Выборочная дисперсия $D_{\textit{в}}$     |            0.715126 |                 0.715126 | True         |
| Выборочное СКО $\sigma_{\textit{в}}$      |            0.845651 |                 0.845651 | True         |

In [ ]:
stats = calculate_sample_stats(width_moments, width_h, width_C)
display_sample_stats(stats)

| Характеристика                            |   Стандартный метод |   Метод условных вариант | Совпадение   |
|:------------------------------------------|--------------------:|-------------------------:|:-------------|
| Выборочное среднее $\bar{x}_{\textit{в}}$ |            3.06121  |                 3.06121  | True         |
| Выборочная дисперсия $D_{\textit{в}}$     |            0.200061 |                 0.200061 | True         |
| Выборочное СКО $\sigma_{\textit{в}}$      |            0.447282 |                 0.447282 | True         |

  - Вычислить исправленную выборочную дисперсию $ s^2 $ и исправленное СКО $ s $. Сравнить данные оценки с смещёнными оценками дисперсии и СКО. Сделать выводы.


In [ ]:
def calculate_corrected_stats(stats: dict[str, float], n: int) -> dict[str, float]:
    """
    Calculates corrected sample variance and standard deviation.
    """
    D_v = stats["D_v_std"]
    sigma_v = stats["sigma_v_std"]
    
    s2 = (n / (n - 1)) * D_v
    s = np.sqrt(s2)
    
    return {
        "D_v": D_v,
        "sigma_v": sigma_v,
        "s2": s2,
        "s": s
    }

def display_corrected_stats(corrected_stats: dict[str, float]) -> None:
    data = [
        ["Дисперсия", r"$D_{\textit{в}}$", corrected_stats['D_v'], r"$s^2$", corrected_stats['s2']],
        ["СКО", r"$\sigma_{\textit{в}}$", corrected_stats['sigma_v'], r"$s$", corrected_stats['s']],
    ]
    
    df_display = pd.DataFrame(data, columns=["Характеристика", "Обозначение (смещ.)", "Значение (смещ.)", "Обозначение (испр.)", "Значение (испр.)"])
    
    display_markdown(df_display.to_markdown(index=False), raw=True)
    
    diff_var = corrected_stats['s2'] - corrected_stats['D_v']
    diff_std = corrected_stats['s'] - corrected_stats['sigma_v']
    
    print(f"Разница дисперсий (s^2 - D_v): {diff_var:.6f}")
    print(f"Разница СКО (s - sigma_v): {diff_std:.6f}")
    print("\n")


In [ ]:
n_length = length_moments["m_i"].sum()
stats_length = calculate_sample_stats(length_moments, length_h, length_C)
corrected_stats_length = calculate_corrected_stats(stats_length, n_length)
display_corrected_stats(corrected_stats_length)


| Характеристика   | Обозначение (смещ.)   |   Значение (смещ.) | Обозначение (испр.)   |   Значение (испр.) |
|:-----------------|:----------------------|-------------------:|:----------------------|-------------------:|
| Дисперсия        | $D_{\textit{в}}$      |           0.715126 | $s^2$                 |           0.721873 |
| СКО              | $\sigma_{\textit{в}}$ |           0.845651 | $s$                   |           0.849631 |

Разница дисперсий (s^2 - D_v): 0.006746
Разница СКО (s - sigma_v): 0.003980




In [ ]:
n_width = width_moments["m_i"].sum()
stats_width = calculate_sample_stats(width_moments, width_h, width_C)
corrected_stats_width = calculate_corrected_stats(stats_width, n_width)
display_corrected_stats(corrected_stats_width)

| Характеристика   | Обозначение (смещ.)   |   Значение (смещ.) | Обозначение (испр.)   |   Значение (испр.) |
|:-----------------|:----------------------|-------------------:|:----------------------|-------------------:|
| Дисперсия        | $D_{\textit{в}}$      |           0.200061 | $s^2$                 |           0.201949 |
| СКО              | $\sigma_{\textit{в}}$ |           0.447282 | $s$                   |           0.449387 |

Разница дисперсий (s^2 - D_v): 0.001887
Разница СКО (s - sigma_v): 0.002105




В обоих случаях исправленная выборочная дисперсия $s^2$ и исправленное СКО $s$ оказались *больше* соответствующих смещённых оценок ($D_{\textit{в}}$ и $\sigma_{\textit{в}}$). Это согласуется с теоретическими свойствами оценок: выборочная дисперсия $D_{\textit{в}}$ является смещённой оценкой и систематически занижает истинную дисперсию генеральной совокупности, в то время как $s^2 = \frac{n}{n-1} D_{\textit{в}}$ является несмещённой оценкой.

  - Найти статистическую оценку коэффициентов асимметрии $ \bar{A}_s $ и эксцесса $ \bar{E} $. Сделать выводы.


In [ ]:
def calculate_skewness_kurtosis(moments: pd.DataFrame, stats: dict[str, float]) -> dict[str, float]:
    """
    Calculates sample skewness and kurtosis.
    """
    sigma = stats["sigma_v"]
    m3 = moments[moments["k"] == 3]["m*"].values[0]
    m4 = moments[moments["k"] == 4]["m*"].values[0]
    
    A_s = m3 / (sigma ** 3)
    
    E = m4 / (sigma ** 4) - 3
    
    return {
        "A_s": A_s,
        "E": E
    }

def display_skewness_kurtosis(metrics: dict[str, float]) -> None:
    data = [
        [r"Асимметрия $\bar{A}_s$", metrics['A_s']],
        [r"Эксцесс $\bar{E}$", metrics['E']],
    ]
    
    df_display = pd.DataFrame(data, columns=["Характеристика", "Значение"])
    
    display_markdown(df_display.to_markdown(index=False), raw=True)

In [ ]:
skew_kurt_length = calculate_skewness_kurtosis(moments_metrics_length, corrected_stats_length)
display_skewness_kurtosis(skew_kurt_length)

| Характеристика         |   Значение |
|:-----------------------|-----------:|
| Асимметрия $\bar{A}_s$ |   0.282135 |
| Эксцесс $\bar{E}$      |  -0.700427 |

In [ ]:
skew_kurt_width = calculate_skewness_kurtosis(moments_metrics_width, corrected_stats_width)
display_skewness_kurtosis(skew_kurt_width)

| Характеристика         |   Значение |
|:-----------------------|-----------:|
| Асимметрия $\bar{A}_s$ |   0.308272 |
| Эксцесс $\bar{E}$      |  -0.133334 |

Таким образом, распределения обоих признаков имеют *правостороннюю асимметрию* (положительное значение $ \bar{A}_s $) и *плосковершинное* распределение (отрицательнок значение $ \bar{E} $). Это означает, что данные имеют длинный левый хвост и более выраженный пик по сравнению с нормальным распределением.

  - Для интервального ряда вычислить моду $ M^*_o $, медиану $ M^*_e $ и коэффициент вариации $ V^* $ заданного распределения. Сделать выводы.

In [ ]:
def calculate_structural_averages(df_intervals: pd.DataFrame, h: float, stats: dict[str, float]) -> dict[str, float]:
    """
    Calculates structural averages (Mode, Median) and Coefficient of Variation.
    """
    n = df_intervals["m_i"].sum()
    
    mode_idx = df_intervals["m_i"].idxmax()
    if isinstance(mode_idx, str):
        raise ValueError("Mode index should be an integer index, got string.")
    m_Mo = df_intervals["m_i"].iloc[mode_idx]
    
    m_Mo_prev = df_intervals["m_i"].iloc[mode_idx - 1] if mode_idx > 0 else 0
    m_Mo_next = df_intervals["m_i"].iloc[mode_idx + 1] if mode_idx < len(df_intervals) - 1 else 0
    
    x_Mo_min = df_intervals["x_i"].iloc[mode_idx] - h / 2
    
    denominator_mo = (m_Mo - m_Mo_prev) + (m_Mo - m_Mo_next)
    if denominator_mo == 0:
         Mode = x_Mo_min + h * 0.5
    else:
        Mode = x_Mo_min + h * ((m_Mo - m_Mo_prev) / denominator_mo)
    
    median_idx = df_intervals[df_intervals["m_i_cum"] > n / 2].index[0]
    
    x_Me_min = df_intervals["x_i"].iloc[median_idx] - h / 2
    S_Me_prev = df_intervals["m_i_cum"].iloc[median_idx - 1] if median_idx > 0 else 0
    m_Me = df_intervals["m_i"].iloc[median_idx]
    
    Median = x_Me_min + h * ((0.5 * n - S_Me_prev) / m_Me)
    
    x_bar = stats["x_bar_std"]
    sigma = stats["sigma_v_std"]
    V_star = (sigma / x_bar) * 100
    
    return {
        "Mode": Mode,
        "Median": Median,
        "V_star": V_star
    }

def display_structural_averages(metrics: dict[str, float]) -> None:
    data = [
        [r"Мода $M^*_o$", metrics['Mode']],
        [r"Медиана $M^*_e$", metrics['Median']],
        [r"Коэффициент вариации $V^*$", f"{metrics['V_star']:.2f}%"]
    ]
    
    df_display = pd.DataFrame(data, columns=["Характеристика", "Значение"])
    display_markdown(df_display.to_markdown(index=False), raw=True)

In [ ]:
avgs_length = calculate_structural_averages(length_intervals, length_h, stats_length)
display_structural_averages(avgs_length)

| Характеристика             | Значение          |
|:---------------------------|:------------------|
| Мода $M^*_o$               | 6.242105263157895 |
| Медиана $M^*_e$            | 5.901470588235294 |
| Коэффициент вариации $V^*$ | 14.28%            |

In [ ]:
avgs_width = calculate_structural_averages(width_intervals, width_h, stats_width)
display_structural_averages(avgs_width)

| Характеристика             | Значение           |
|:---------------------------|:-------------------|
| Мода $M^*_o$               | 2.82               |
| Медиана $M^*_e$            | 3.0239130434782613 |
| Коэффициент вариации $V^*$ | 14.61%             |

## Выводы

В ходе выполнения работы были получены точечные оценки параметров распределения для двух количественных признаков: длина чашелистика (SepalLengthCm) и ширина чашелистика (SepalWidthCm).

Выборочное среднее ($\bar{x}_{\textit{в}}$) и выборочная дисперсия ($D_{\textit{в}}$), рассчитанные стандартным методом и методом условных вариант, совпали, что подтверждает корректность проведенных вычислений.

Исправленная выборочная дисперсия ($s^2$) и исправленное среднее квадратическое отклонение ($s$) оказались больше смещённых оценок ($D_{\textit{в}}$, $\sigma_{\textit{в}}$), что соответствует теоретическим ожиданиям для конечных выборок ($n/(n-1) > 1$).

Коэффициенты асимметрии ($\bar{A}_s$) для обоих признаков положительны (0.28 и 0.31), что указывает на незначительную правостороннюю асимметрию распределения.
Коэффициенты эксцесса ($\bar{E}$) отрицательны (-0.70 и -0.13), что свидетельствует о плосковершинном характере распределения (кривая распределения более пологая по сравнению с нормальным).

Коэффициенты вариации ($V^*$) для длины и ширины чашелистика составили 14.28% и 14.61% соответственно. Так как эти значения не превышают 33%, исследуемая совокупность является однородной по данным признакам, а выборочная средняя — надежной характеристикой.

Рассчитанные значения моды ($M^*_o$) и медианы ($M^*_e$) для интервальных рядов позволяют уточнить характер распределения и его центральную тенденцию.